# Detección de Fraude en Tarjetas de Crédito

Proyecto de clasificación (desbalanceado) listo para **Google Colab**.

**Pregunta:** ¿podemos detectar transacciones fraudulentas?

**Target:** `Class` (1 = fraude, 0 = normal)

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico de credit card fraud.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, average_precision_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
df = pd.read_csv(url)
df = df.sample(n=min(20000, len(df)), random_state=RANDOM_STATE)
print(df.shape)
print(df["Class"].value_counts())
df.head()

## 2. EDA

In [ ]:
print("Tasa de fraude:", df["Class"].mean().round(4))
df.describe().T.head(10)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Class")
plt.xticks([0, 1], ["Normal", "Fraude"])
plt.title("Distribución (muy desbalanceada)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="Amount", hue="Class", bins=50, log_scale=True)
plt.title("Amount por clase (log scale)")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
corr = df.corr()["Class"].drop("Class").sort_values()
corr.plot(kind="barh")
plt.title("Correlación con Class")
plt.show()

### Hallazgos
- Extremadamente desbalanceado (<0.2% fraude).
- Features V1–V28 son PCA.
- Amount y Time también son útiles.

## 3. Preparar datos

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("pr_auc", ascending=False)

## 5. Evaluación

In [ ]:
mejor = "Random Forest"
pipe = pipes[mejor]
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Normal", "Fraude"]))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Fraude"]).plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()

## 6. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)

plt.figure(figsize=(8, 5))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Top features — Random Forest")
plt.tight_layout()
plt.show()

## 7. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled[:200])

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values[1], X_train_scaled[:200], feature_names=X.columns, max_display=10, show=False)
plt.title("SHAP Summary — Fraude")
plt.tight_layout()
plt.show()

## 8. Conclusiones

- Dataset muy desbalanceado: mejor mirar PR-AUC y recall de la clase fraude.
- ROC-AUC suele ser alto (>0.95), pero PR-AUC es más informativo.
- Features PCA (V14, V12, V17, etc.) suelen ser las más importantes.

**Rango esperado:** ROC-AUC ~0.95–0.99 | PR-AUC ~0.70–0.85